# 02c. Диагностика прогнозов: где модели ошибаются и насколько

Графики по результатам `02_backtest` (`results/predictions_all.parquet`). Все картинки сохраняются
в `results/figures/`. Везде, где возможно, метрики считаются по **всем МО**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = ROOT / "results"; FIG = RES / "figures"; FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

P = pd.read_parquet(RES / "predictions_all.parquet")
panel = pd.read_parquet(ROOT / "data" / "processed" / "panel_mo_month.parquet")
info = panel.drop_duplicates("mo_id").set_index("mo_id")[["mo_name_sber", "region", "size_class", "population"]]
P = P.join(info, on="mo_id").dropna(subset=["y_true"])
P["target_date"] = (P.origin.dt.to_period("M") + P.h.astype(int)).dt.to_timestamp()
P["target_month"] = P.target_date.dt.month
if "prophet_all" in P: P["prophet"] = P.prophet_all
for k, cols in {"ens_snt_lgbm": ["snt", "lgbm_resid"], "ens_snt_lgbm_logratio": ["snt", "lgbm_logratio"],
                "ens_prophet_lgbm": ["prophet", "lgbm_resid"]}.items():
    if all(c in P for c in cols): P[k] = P[cols].mean(axis=1, skipna=False)
LABEL = {"naive": "Naive", "snt": "Сезонный бейзлайн (МО)", "snt_nat": "Сезонный бейзлайн (страна)", "prophet": "Prophet",
         "prophet_tuned": "Prophet усиленный", "lgbm_logratio": "LightGBM (лог-отношение)", "lgbm_resid": "LightGBM (поправка к бейзлайну)",
         "lgbm_resid_nonat": "LightGBM без нац. признаков", "lgbm_resid_nonat_news": "LightGBM + новости",
         "ens_snt_lgbm": "Ансамбль: бейзлайн + LightGBM", "ens_snt_lgbm_logratio": "Ансамбль: бейзлайн + LightGBM (лог-отн.)",
         "ens_prophet_lgbm": "Ансамбль: Prophet + LightGBM"}
MODELS = [m for m in LABEL if m in P and P[m].notna().any()]
KEY = [m for m in ["prophet", "snt", "snt_nat", "lgbm_resid", "ens_snt_lgbm"] if m in MODELS]
H_MAIN = [1, 3, 6, 12]
def mae(d, m): x = d.dropna(subset=[m]); return (x[m] - x.y_true).abs().mean() if len(x) else np.nan
def mape(d, m): x = d.dropna(subset=[m]); return ((x[m] - x.y_true).abs() / x.y_true).mean() * 100 if len(x) else np.nan
print("моделей:", len(MODELS), "| строк:", len(P), "| МО:", P.mo_id.nunique())

## 1. Сравнение моделей по горизонтам
Столбики — MAE по всем МО и отсечкам; серая пунктирная линия — Prophet. Чем короче столбик, тем лучше.

In [ ]:
tab = pd.DataFrame({h: {m: mae(P[P.h == h], m) for m in MODELS} for h in H_MAIN})
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5), sharey=False)
for ax, h in zip(axes, H_MAIN):
    s = tab[h].dropna().sort_values()
    colors = ["#d62728" if m == "prophet" else ("#2ca02c" if m.startswith("ens") else "#1f77b4") for m in s.index]
    ax.barh([LABEL[m] for m in s.index], s.values, color=colors)
    if "prophet" in s: ax.axvline(s["prophet"], color="gray", ls="--", lw=1)
    ax.set_title(f"h = {h} мес.", fontsize=11); ax.set_xlabel("MAE, руб./жителя")
    for i, v in enumerate(s.values): ax.text(v, i, f" {v:.0f}", va="center", fontsize=8)
fig.suptitle("MAE по моделям и горизонтам (все МО). Красный — Prophet, зелёный — ансамбли", y=1.02)
fig.tight_layout(); fig.savefig(FIG / "01_mae_by_model_horizon.png", bbox_inches="tight"); plt.show()
tab.round(0).to_csv(RES / "diag_mae_table.csv")

## 2. Распределение ошибок
Слева — распределение процентной ошибки на месяц вперёд: у хорошей модели пик узкий и центр в нуле.
Справа — как растёт разброс с горизонтом (ящики: медиана и квартили, усы — 5–95%).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5))
for m in KEY:
    x = P[P.h == 1].dropna(subset=[m]); pe = ((x[m] / x.y_true - 1) * 100).clip(-40, 40)
    a1.hist(pe, bins=80, histtype="step", lw=1.6, label=f"{LABEL[m]} (медиана {pe.median():+.1f}%)")
a1.axvline(0, color="k", lw=0.8); a1.set_xlabel("ошибка прогноза, % от факта (h = 1)"); a1.set_ylabel("число МО × отсечек"); a1.legend(fontsize=8)
best = "ens_snt_lgbm" if "ens_snt_lgbm" in KEY else KEY[-1]
data = [((P[P.h == h][best] / P[P.h == h].y_true - 1) * 100).dropna() for h in [1, 2, 3, 4, 6, 7, 12] if (P.h == h).any()]
a2.boxplot(data, labels=[str(h) for h in [1, 2, 3, 4, 6, 7, 12] if (P.h == h).any()], whis=(5, 95), showfliers=False)
a2.axhline(0, color="k", lw=0.8); a2.set_xlabel("горизонт, мес."); a2.set_ylabel("ошибка, %"); a2.set_title(LABEL[best])
fig.tight_layout(); fig.savefig(FIG / "02_error_distribution.png", bbox_inches="tight"); plt.show()

## 3. Ошибка по отсечкам: когда модели ломаются
Каждая точка — MAE прогноза, сделанного в этот месяц. Всплеск у ноября на h=1 — это прогноз декабря.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, h in zip(axes, [1, 3, 6]):
    d = P[P.h == h]
    for m in KEY:
        s = d.groupby("origin").apply(lambda g: mae(g, m)); ax.plot(s.index, s.values, marker="o", ms=4, label=LABEL[m])
    ax.set_title(f"h = {h}"); ax.set_xlabel("месяц отсечки"); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("MAE, руб."); axes[0].legend(fontsize=8)
fig.suptitle("MAE по отсечкам", y=1.02); fig.tight_layout(); fig.savefig(FIG / "03_mae_by_origin.png", bbox_inches="tight"); plt.show()

## 4. Ошибка по месяцу цели: сезонность ошибок
Тепловая карта MAPE по месяцу, на который делается прогноз. Декабрь — главная проблема всех моделей, кроме сезонных.

In [ ]:
rows = []
for m in KEY:
    for mth, g in P[P.h.isin([1, 2, 3])].groupby("target_month"): rows.append({"модель": LABEL[m], "месяц": mth, "MAPE": mape(g, m)})
hm = pd.DataFrame(rows).pivot(index="модель", columns="месяц", values="MAPE")
fig, ax = plt.subplots(figsize=(13, 3.5))
im = ax.imshow(hm.values, cmap="YlOrRd", aspect="auto"); ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns)
ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels(hm.index); ax.grid(False)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]): ax.text(j, i, f"{hm.values[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("MAPE, % по месяцу цели (горизонты 1–3)"); ax.set_xlabel("месяц, на который делается прогноз")
fig.colorbar(im, ax=ax, fraction=0.02); fig.tight_layout(); fig.savefig(FIG / "04_mape_by_target_month.png", bbox_inches="tight"); plt.show()

## 5. Факт и прогноз: где систематические промахи
Логарифмические оси, цвет — класс размера МО. Точки ниже диагонали — модель занизила.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, m in zip(axes, [k for k in ["prophet", "snt", best] if k in P]):
    d = P[P.h == 1].dropna(subset=[m])
    for cls, g in d.groupby("size_class"): ax.scatter(g.y_true, g[m], s=3, alpha=0.4, label=cls)
    lim = [d.y_true.min() * 0.9, d.y_true.max() * 1.1]; ax.plot(lim, lim, "k--", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("факт, руб."); ax.set_ylabel("прогноз, руб."); ax.set_title(f"{LABEL[m]}, h = 1")
axes[0].legend(fontsize=7, markerscale=4); fig.tight_layout(); fig.savefig(FIG / "05_pred_vs_actual.png", bbox_inches="tight"); plt.show()

## 6. Ошибка по классу размера МО и по регионам

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 5))
order = ["<10k", "10-50k", "50-250k", "250k-1M", ">1M"]
w = 0.8 / len(KEY)
for i, m in enumerate(KEY):
    vals = [mape(P[(P.h == 3) & (P.size_class == c)], m) for c in order]
    a1.bar(np.arange(len(order)) + i * w, vals, w, label=LABEL[m])
a1.set_xticks(np.arange(len(order)) + 0.4 - w / 2); a1.set_xticklabels(order); a1.set_ylabel("MAPE, %"); a1.set_title("по классу размера МО (h = 3)"); a1.legend(fontsize=8)
reg = P[P.h.isin([1, 3])].groupby("region").apply(lambda g: pd.Series({m: mape(g, m) for m in [best, "prophet"] if m in g}))
reg = reg.dropna().sort_values(best, ascending=False).head(20)
a2.barh(reg.index, reg[best], color="#2ca02c", label=LABEL[best])
if "prophet" in reg: a2.scatter(reg["prophet"], reg.index, color="#d62728", s=18, label="Prophet", zorder=3)
a2.invert_yaxis(); a2.set_xlabel("MAPE, % (h = 1–3)"); a2.set_title("20 регионов с наибольшей ошибкой"); a2.legend(fontsize=8); a2.tick_params(axis="y", labelsize=8)
fig.tight_layout(); fig.savefig(FIG / "06_error_by_size_region.png", bbox_inches="tight"); plt.show()

## 7. Самые трудные МО
Что объединяет МО с наибольшей ошибкой: малые, с резкими скачками трат. Полезно для разбора кейсов.

In [ ]:
worst = (P[P.h.isin([1, 3])].groupby("mo_id").apply(lambda g: pd.Series({"MAPE": mape(g, best), "n": len(g)}))
         .join(info).sort_values("MAPE", ascending=False).head(15))
print(worst[["mo_name_sber", "region", "size_class", "MAPE"]].round(1).to_string())
worst.to_csv(RES / "diag_worst_mo.csv")
yy = panel[panel.date >= "2023-01-01"].pivot(index="mo_id", columns="date", values="y_all")
fig, axes = plt.subplots(2, 2, figsize=(15, 7))
T0 = pd.Timestamp("2024-06-01")
for ax, mo in zip(axes.ravel(), list(worst.index[:2]) + list(P[P.h == 1].groupby("mo_id").apply(lambda g: mape(g, best)).sort_values().index[:2])):
    ax.plot(yy.columns, yy.loc[mo], "k-o", ms=3, label="факт")
    for m, c in [("prophet", "#d62728"), ("snt", "#1f77b4"), (best, "#2ca02c")]:
        if m not in P: continue
        path = P[(P.mo_id == mo) & (P.origin == T0)].sort_values("h")
        ax.plot(path.target_date, path[m], "--", color=c, marker=".", label=LABEL[m])
    ax.axvline(T0, color="gray", ls=":", lw=1); ax.set_title(f"{info.loc[mo, 'mo_name_sber'][:45]} ({info.loc[mo, 'size_class']})", fontsize=9)
axes[0, 0].legend(fontsize=7); fig.suptitle("Два самых трудных и два самых простых МО: факт и прогнозы с отсечки июнь 2024", y=1.01)
fig.tight_layout(); fig.savefig(FIG / "07_example_series.png", bbox_inches="tight"); plt.show()

## 8. Остатки против прогноза и вклад новостей
Слева: разброс ошибки должен не зависеть от уровня прогноза; воронка означает, что ошибка растёт с масштабом.
Справа: прирост качества от новостных признаков по горизонтам (отрицательно — новости помогают).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5))
d = P[P.h == 1].dropna(subset=[best]); a1.scatter(d[best], (d.y_true - d[best]) / d[best] * 100, s=3, alpha=0.3)
a1.axhline(0, color="k", lw=0.8); a1.set_xscale("log"); a1.set_xlabel("прогноз, руб."); a1.set_ylabel("остаток, % от прогноза"); a1.set_title(f"{LABEL[best]}, h = 1")
if "lgbm_resid_nonat_news" in P and "lgbm_resid_nonat" in P:
    hs = [h for h in [1, 2, 3, 4, 6, 7] if (P.h == h).any()]
    delta = [mae(P[P.h == h], "lgbm_resid_nonat_news") - mae(P[P.h == h], "lgbm_resid_nonat") for h in hs]
    a2.bar([str(h) for h in hs], delta, color=["#2ca02c" if v < 0 else "#d62728" for v in delta])
    a2.axhline(0, color="k", lw=0.8); a2.set_xlabel("горизонт, мес."); a2.set_ylabel("ΔMAE (с новостями − без), руб."); a2.set_title("Вклад новостей GDELT")
else:
    a2.text(0.5, 0.5, "вариант с новостями не считался", ha="center"); a2.axis("off")
fig.tight_layout(); fig.savefig(FIG / "08_residuals_and_news.png", bbox_inches="tight"); plt.show()